# LA Food Scout — Exploratory Data Analysis & Methodology

This notebook documents the full data exploration and modeling process for the LA Food Scout project.

**Goal**: Predict whether an LA restaurant is likely to be **highly rated (≥ 4.5★)** based on features available without visiting the restaurant.

**Data source**: Google Places API (Nearby Search) across 20 LA neighborhoods, collected May 2026.

## 1. Setup

In [ ]:
import pathlib
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

DATA_PATH = pathlib.Path('../data/la_restaurants.csv')
MODEL_PATH = pathlib.Path('../models/model.pkl')

df = pd.read_csv(DATA_PATH)
print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Numeric Summary ===')
df.describe().round(2)

In [ ]:
# Drop rows with missing rating (these have no label)
df = df.dropna(subset=['rating'])
df['high_rated'] = (df['rating'] >= 4.5).astype(int)
df['review_count_log'] = np.log1p(df['review_count'])
df['price_level'] = df['price_level'].fillna(df['price_level'].median())

print(f'Clean dataset: {len(df):,} restaurants')
print(f'High-rated (>=4.5): {df["high_rated"].sum():,} ({df["high_rated"].mean()*100:.1f}%)')
print(f'Neighborhoods: {df["neighborhood"].nunique()}')
print(f'Place types:   {df["primary_type"].nunique()}')

## 3. Rating Distribution

The dataset shows a left-skewed distribution centered around 4.3–4.4, which is typical of Google Places data — only active, reviewed businesses tend to have ratings. The 4.5 threshold captures roughly the top 41% of restaurants.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Rating histogram
axes[0].hist(df['rating'], bins=25, color='#FF6B6B', edgecolor='white')
axes[0].axvline(4.5, color='black', linestyle='--', linewidth=1.5, label='4.5★ threshold')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].legend()

# Class balance
counts = df['high_rated'].value_counts()
axes[1].bar(['Regular (<4.5★)', 'High-Rated (≥4.5★)'], counts.values,
            color=['#74B9FF', '#FD79A8'])
axes[1].set_title('Class Balance')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Class ratio — Regular: {counts[0]} | High-rated: {counts[1]}')

## 4. Feature Analysis: Price Level

Price level is a 1–4 ordinal scale from Google Places (1=Budget, 4=Very Expensive). We check whether price correlates with rating quality.

In [ ]:
price_labels = {1.0: '$\nBudget', 2.0: '$$\nModerate', 3.0: '$$$\nExpensive', 4.0: '$$$$\nVery Exp.'}
price_data = df.dropna(subset=['price_level'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Avg rating by price
price_avg = price_data.groupby('price_level')['rating'].mean()
x_labels = [price_labels.get(k, str(k)) for k in price_avg.index]
bars = axes[0].bar(range(len(x_labels)), price_avg.values,
                   color=['#4ECDC4','#45B7D1','#96CEB4','#FFEAA7'][:len(x_labels)])
axes[0].set_xticks(range(len(x_labels)))
axes[0].set_xticklabels(x_labels)
axes[0].set_title('Avg Rating by Price Level')
axes[0].set_ylim(4.0, 4.8)
for bar, val in zip(bars, price_avg.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', fontsize=9)

# % high-rated by price
price_pct = price_data.groupby('price_level')['high_rated'].mean() * 100
x_labels2 = [price_labels.get(k, str(k)) for k in price_pct.index]
axes[1].bar(range(len(x_labels2)), price_pct.values, color='#A29BFE')
axes[1].set_xticks(range(len(x_labels2)))
axes[1].set_xticklabels(x_labels2)
axes[1].set_title('% High-Rated by Price Level')
axes[1].set_ylabel('% of Restaurants')

plt.tight_layout()
plt.show()

## 5. Feature Analysis: Neighborhood

In [ ]:
hood_stats = df.groupby('neighborhood').agg(
    count=('rating', 'count'),
    avg_rating=('rating', 'mean'),
    pct_high=('high_rated', 'mean')
).sort_values('avg_rating', ascending=False)
hood_stats['pct_high'] = (hood_stats['pct_high'] * 100).round(1)
hood_stats['avg_rating'] = hood_stats['avg_rating'].round(3)
print(hood_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

hood_avg = df.groupby('neighborhood')['rating'].mean().sort_values()
axes[0].barh(hood_avg.index, hood_avg.values, color='#FD79A8')
axes[0].set_title('Avg Rating by Neighborhood')
axes[0].set_xlabel('Avg Rating')
axes[0].set_xlim(4.0, 4.7)
for i, (hood, val) in enumerate(hood_avg.items()):
    axes[0].text(val + 0.005, i, f'{val:.2f}', va='center', fontsize=8)

high_pct = df.groupby('neighborhood')['high_rated'].mean().sort_values() * 100
axes[1].barh(high_pct.index, high_pct.values, color='#FDCB6E')
axes[1].set_title('% High-Rated (≥4.5★) by Neighborhood')
axes[1].set_xlabel('% of Restaurants')
for i, (hood, val) in enumerate(high_pct.items()):
    axes[1].text(val + 0.5, i, f'{val:.0f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()

## 6. Feature Analysis: Place Type & Review Count

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Top 10 place types
top_types = df['primary_type'].value_counts().head(10)
axes[0].barh(top_types.index[::-1], top_types.values[::-1], color='#55EFC4')
axes[0].set_title('Top 10 Place Types')
axes[0].set_xlabel('Count')

# Log review count vs rating scatter
axes[1].scatter(df['review_count_log'], df['rating'],
                alpha=0.25, color='#A29BFE', s=15)
axes[1].set_title('Log(Review Count) vs Rating')
axes[1].set_xlabel('log(review_count + 1)')
axes[1].set_ylabel('Rating')

# Trend line
z = np.polyfit(df['review_count_log'].dropna(), df.loc[df['review_count_log'].notna(), 'rating'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['review_count_log'].min(), df['review_count_log'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=1.5, label=f'trend')
axes[1].legend()

plt.tight_layout()
plt.show()

corr = df[['rating', 'review_count_log', 'price_level']].corr()
print('Feature correlations with rating:')
print(corr['rating'].drop('rating').round(4))

## 7. Modeling Approach

### Problem Framing
Binary classification: predict `high_rated` (1 if rating ≥ 4.5, else 0).

### Features
| Feature | Type | Rationale |
|---|---|---|
| `price_level` | Numeric (1–4) | Price tier as a proxy for quality/curation |
| `review_count_log` | Numeric | Log-scaled to reduce skew; popularity signal |
| `neighborhood` | Categorical | Area-level quality variation |
| `primary_type` | Categorical | Different cuisines/formats have different rating baselines |

### Why Random Forest?
- Handles mixed numeric/categorical features well via one-hot encoding
- `class_weight='balanced'` addresses the 59/41 class imbalance without oversampling
- Non-linear interactions (e.g., expensive restaurants in Brentwood behave differently than expensive restaurants in Downtown)
- Feature importance is interpretable

### Preprocessing Pipeline
- `StandardScaler` on numeric features
- `OneHotEncoder(handle_unknown='ignore')` on categorical features
- Entire pipeline bundled in `model.pkl` so no separate preprocessor is needed at inference time

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, roc_auc_score,
    ConfusionMatrixDisplay, RocCurveDisplay
)

NUMERIC = ['price_level', 'review_count_log']
CATEGORICAL = ['neighborhood', 'primary_type']

X = df[NUMERIC + CATEGORICAL]
y = df['high_rated']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

bundle = joblib.load(MODEL_PATH)
pipeline = bundle['pipeline']

y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print('=== Test Set Performance ===')
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Regular', 'High-Rated']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Regular', 'High-Rated'],
    ax=axes[0], colorbar=False
)
axes[0].set_title('Confusion Matrix')

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].set_title('ROC Curve')
axes[1].plot([0,1],[0,1],'k--')

plt.tight_layout()
plt.show()

In [ ]:
# Feature importances from the Random Forest
rf = pipeline.named_steps['classifier']
ohe = pipeline.named_steps['preprocessor'].named_transformers_['cat']
cat_feature_names = ohe.get_feature_names_out(CATEGORICAL).tolist()
all_feature_names = NUMERIC + cat_feature_names

importances = pd.Series(rf.feature_importances_, index=all_feature_names)
top20 = importances.nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top20.index, top20.values, color='#74B9FF')
ax.set_title('Top 20 Feature Importances (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Key Findings & Conclusions

### What the model learned
- **Review count** (log-scaled) is the strongest single predictor — establishments with more reviews tend to have more stable, high ratings
- **Neighborhood** has meaningful signal: some areas (e.g., Monterey Park, Los Feliz) have higher concentrations of top-rated spots
- **Price level** alone is a weak predictor; moderate-priced restaurants can be just as highly rated as expensive ones
- **Place type** contributes meaningfully — bakeries and cafes have different rating distributions than general restaurants

### Model Limitations
- ROC-AUC of 0.65 — better than random (0.5) but not highly accurate. Restaurant ratings are inherently noisy (subjective, culturally influenced)
- The model has no information about food quality, service, ambiance — the features available are all "meta" signals
- All data comes from one API snapshot; ratings drift over time

### Future improvements
- Incorporate text features from reviews (sentiment, keywords)
- Add geographic proximity features (distance to landmarks, transit)
- Experiment with gradient boosting with tuned hyperparameters
- Collect longitudinal data to model rating trajectory